In [2]:
!!pip install litellm

# Important!!!
#
# <---- Set your 'GEMINI_API_KEY' as a secret over there with the "key" icon
#
#
import os
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')
os.environ['GEMINI_API_KEY'] = api_key

In [20]:
from google import genai
from typing import List,Dict

client=genai.Client()
import json
from google.genai import types



def list_files()-> List[str]:
  return os.listdir(".")

def read_file(file_name:str)->str:
  try :
     with open(file_name, "r") as file:
        return file.read()
  except FileNotFoundError:
    return f"Error: {file_name} not found"
  except Exception as e:
    return f"Error: {str(e)}"

tool_function={
    "list_files":list_files,
    "read_file":read_file
}

tools = [
    types.Tool(
        function_declarations=[
            types.FunctionDeclaration(
                name="list_files",
                description="Returns a list of files in the directory.",
                parameters_json_schema={
                    "type": "object",
                    "properties": {},
                    "required": []
                }
            ),
            types.FunctionDeclaration(
                name="read_file",
                description="Reads a specified file.",
                parameters_json_schema={
                    "type": "object",
                    "properties": {
                        "file_name": {"type": "string"}
                    },
                    "required": ["file_name"]
                }
            )
        ]
    )
]

agent_rules=[{
    "role":"system",
    "content":"""
    You are an AI agent that can perform tasks by using available tools.

If a user asks about files, documents, or content, first list the files before reading them.
    """
}]


user_task = input("What would you like me to do? ")

memory = [{"role": "user", "content": user_task}]

messages = agent_rules + memory

response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=user_task,

        config=types.GenerateContentConfig(
         system_instruction=agent_rules[0]["content"],
         tools=tools,
         max_output_tokens=1024
        )

    )
if response.function_calls:

    tool_call = response.function_calls[0]

    tool_name = tool_call.name
    tool_args = tool_call.args

    print("Tool Name:", tool_name)
    print("Tool Arguments:", tool_args)

    result = tool_functions[tool_name](**tool_args)

    print("Result:", result)

else:
    print(response.text)


What would you like me to do? give me the files name in the directory
Tool Name: list_files
Tool Arguments: {}
Result: ['.config', 'sample_data']
